# Cálculo de KPIs en Python

Este notebook replica las consultas del fichero `Calculo KPIs.sql` usando **pandas**.

**Requisito:** debe existir el fichero `datos/tablas.pkl`. Si no lo tienes, ejecuta antes el **paso 2** del notebook `EDA_Jupiter.ipynb`.

## 1. Importar librerías

In [1]:
import pandas as pd

## 2. Cargar tablas en DataFrames

Igual que el **paso 4** de `EDA_Jupiter.ipynb`: leemos todas las tablas desde el fichero `datos/tablas.pkl`.

In [2]:
from DB import cargar_datos

datos = cargar_datos("datos/tablas.pkl")

print(f"Tablas cargadas desde fichero: {len(datos)}")
for nombre, df in datos.items():
    print(f"  - {nombre}: {df.shape[0]} filas x {df.shape[1]} columnas")

Tablas cargadas desde fichero: 8
  - cliente: 34 filas x 2 columnas
  - lote: 69567 filas x 2 columnas
  - marca: 35 filas x 2 columnas
  - producto: 70549 filas x 9 columnas
  - proveedor: 35 filas x 2 columnas
  - subtipo: 27 filas x 3 columnas
  - tipo: 15 filas x 2 columnas
  - venta: 70549 filas x 6 columnas


### 2.1 Crear variables para cada tabla

In [3]:
df_cliente = datos["cliente"]
df_lote = datos["lote"]
df_marca = datos["marca"]
df_producto = datos["producto"]
df_proveedor = datos["proveedor"]
df_subtipo = datos["subtipo"]
df_tipo = datos["tipo"]
df_venta = datos["venta"]

### 2.2 Preparar columnas útiles

In [4]:
# Copias para no modificar los DataFrames originales
venta = df_venta.copy()
producto = df_producto.copy()

# Fechas de venta
venta["tiempo_venta"] = pd.to_datetime(venta["tiempo_venta"])
venta["fecha_dia"] = venta["tiempo_venta"].dt.date
venta["mes"] = venta["tiempo_venta"].dt.month
venta["dia_mes"] = venta["tiempo_venta"].dt.day

# Valor de cada venta (precio por kilo * peso en gramos / 1000)
venta["valor_euros"] = venta["precio_venta"] * venta["peso"] / 1000

# Fechas de recogida
producto["tiempo_recogida"] = pd.to_datetime(producto["tiempo_recogida"])
producto["mes_recogida"] = producto["tiempo_recogida"].dt.month
producto["hora_recogida"] = producto["tiempo_recogida"].dt.hour

print("Datos preparados correctamente.")

Datos preparados correctamente.


---
## KPI 1) Media diaria de la cuantía de las distribuciones

**Interpretación 1:** número medio de distribuciones por día (solo ventas con fecha válida).

In [5]:
# Unir venta con producto (INNER JOIN)
venta_producto = venta.merge(producto, on="id_producto")

# Contar distribuciones por día
por_dia = (
    venta_producto.groupby("fecha_dia", as_index=False)
    .agg(nro_distribuciones=("id_lote", "count"))
)

# Solo días con fecha válida
por_dia = por_dia[por_dia["fecha_dia"].notna()]

media_distribuciones_diaria = round(por_dia["nro_distribuciones"].mean(), 2)
print(f"Media diaria de distribuciones: {media_distribuciones_diaria}")

Media diaria de distribuciones: 2197.69


**Interpretación 2:** si cuantía = valor económico de las ventas (precio por kilo).

In [6]:
valor_por_dia = (
    venta.groupby("fecha_dia", as_index=False)
    .agg(valor_venta=("valor_euros", "sum"))
)

valor_por_dia = valor_por_dia[valor_por_dia["fecha_dia"].notna()]

cuantia_media_diaria = round(valor_por_dia["valor_venta"].mean(), 2)
print(f"Cuantía media diaria (euros): {cuantia_media_diaria}")

Cuantía media diaria (euros): 2287.04


---
## KPI 2) Cuantía total de las distribuciones

In [7]:
total_distribuciones = por_dia["nro_distribuciones"].sum()
print(f"Total distribuciones (suma por días): {total_distribuciones}")

total_lotes_distintos = producto["id_lote"].nunique()
print(f"Total lotes distintos en producto: {total_lotes_distintos}")

Total distribuciones (suma por días): 70326
Total lotes distintos en producto: 69567


In [8]:
cuantia_total = round(valor_por_dia["valor_venta"].sum(), 2)
print(f"Cuantía total (euros): {cuantia_total}")

Cuantía total (euros): 73185.3


---
## KPI 3) Días del mes con más distribuciones (septiembre)

Top 5 días del mes con más distribuciones.

In [9]:
venta_sep = venta_producto[venta_producto["mes"] == 9]

dias_mas_distribuciones = (
    venta_sep.groupby("dia_mes", as_index=False)
    .agg(nro_distribuciones=("id_lote", "count"))
    .sort_values("nro_distribuciones", ascending=False)
    .head(5)
)

dias_mas_distribuciones.columns = ["fecha_dia", "nro_distribuciones"]
dias_mas_distribuciones

,fecha_dia,nro_distribuciones
12,13.0,2385
13,14.0,2378
2,3.0,2376
17,18.0,2376
21,22.0,2367


---
## KPI 4) Horas del día con más recogidas (septiembre)

Top 5 horas con más recogidas de alimentos.

In [10]:
producto_sep = producto[producto["mes_recogida"] == 9]

horas_mas_recogidas = (
    producto_sep.groupby("hora_recogida", as_index=False)
    .agg(nro_de_recogidas=("id_producto", "count"))
    .sort_values("nro_de_recogidas", ascending=False)
    .head(5)
)

horas_mas_recogidas

,hora_recogida,nro_de_recogidas
13,13,3001
18,18,2997
21,21,2995
10,10,2994
19,19,2992


---
## KPI 5) Top 5 clientes que más dinero han gastado (septiembre)

In [11]:
venta_cliente = venta[venta["mes"] == 9].merge(df_cliente, on="id_cliente")

top5_clientes_gasto = (
    venta_cliente.groupby("nombre", as_index=False)
    .agg(gasto_por_cliente=("valor_euros", "sum"))
    .sort_values("gasto_por_cliente", ascending=False)
    .head(5)
)

top5_clientes_gasto["gasto_por_cliente"] = top5_clientes_gasto["gasto_por_cliente"].round(2)
top5_clientes_gasto.columns = ["nombre_cliente", "gasto_por_cliente"]
top5_clientes_gasto

,nombre_cliente,gasto_por_cliente
27,SuperEconómico,2208.13
28,SuperMercado Ideal,2202.00
23,MegaCompra,2166.87
7,Cosecha Fresca,2164.19
9,Distribuciones del Sol,2160.60


---
## KPI 6) Top 5 clientes que menos dinero han gastado (septiembre)

In [12]:
top5_clientes_menos_gasto = (
    venta_cliente.groupby("nombre", as_index=False)
    .agg(gasto_por_cliente=("valor_euros", "sum"))
    .sort_values("gasto_por_cliente", ascending=True)
    .head(5)
)

top5_clientes_menos_gasto["gasto_por_cliente"] = top5_clientes_menos_gasto["gasto_por_cliente"].round(2)
top5_clientes_menos_gasto.columns = ["nombre_cliente", "gasto_por_cliente"]
top5_clientes_menos_gasto

,nombre_cliente,gasto_por_cliente
11,Distribuidora Gourmet,2000.50
2,Almacén Estrella,2015.00
3,Central de Abastos Central,2047.46
5,CompraMaestra,2056.48
12,Distribuidora Nacional,2057.52


---
## KPI 7) Top 10 proveedores que más dinero han recibido (septiembre)

In [13]:
venta_proveedor = (
    venta[venta["mes"] == 9]
    .merge(producto[["id_producto", "id_proveedor"]], on="id_producto")
    .merge(df_proveedor, on="id_proveedor")
)

top10_proveedores = (
    venta_proveedor.groupby("nombre", as_index=False)
    .agg(cuantia_ingreso_proveedor=("valor_euros", "sum"))
    .sort_values("cuantia_ingreso_proveedor", ascending=False)
    .head(10)
)

top10_proveedores["cuantia_ingreso_proveedor"] = top10_proveedores["cuantia_ingreso_proveedor"].round(2)
top10_proveedores.columns = ["nombre_proveedor", "cuantia_ingreso_proveedor"]
top10_proveedores

,nombre_proveedor,cuantia_ingreso_proveedor
13,EcoFungicidas Morales,2108.24
22,Maquinaria Agrícola Avanzada,2107.65
3,AgroSoftware Soluciones,2094.31
20,Insecticidas Ecológicos Reyes,2090.87
32,Tractores Eficientes Delgado,2083.16
7,Apícola EcoMiel,2082.00
0,Agricultura Inteligente TechCultivos,2080.95
5,Agronutrientes del Futuro,2078.25
12,Cultivos Hidropónicos Innovagro,2070.36
25,Productos Lácteos EcoLeche,2068.09


---
## KPI 8) Top 3 productos con mayor beneficio (septiembre)

Beneficio = precio de venta − coste inicial.

In [14]:
venta_tipo = (
    venta[venta["mes"] == 9]
    .merge(producto[["id_producto", "id_tipo", "coste_inicial"]], on="id_producto")
    .merge(df_tipo, on="id_tipo")
)

venta_tipo["margen"] = venta_tipo["precio_venta"] - venta_tipo["coste_inicial"]

top3_mayor_beneficio = (
    venta_tipo.groupby("nombre", as_index=False)
    .agg(margen_producto=("margen", "sum"))
    .sort_values("margen_producto", ascending=False)
    .head(3)
)

top3_mayor_beneficio.columns = ["tipo_fruta", "margen_producto"]
top3_mayor_beneficio

,tipo_fruta,margen_producto
3,Guava,27708.02
0,Apple,15610.90
4,Kiwi,12008.17


---
## KPI 9) Top 3 productos con peor beneficio (septiembre)

In [15]:
top3_peor_beneficio = (
    venta_tipo.groupby("nombre", as_index=False)
    .agg(margen_producto=("margen", "sum"))
    .sort_values("margen_producto", ascending=True)
    .head(3)
)

top3_peor_beneficio.columns = ["tipo_fruta", "margen_producto"]
top3_peor_beneficio

,tipo_fruta,margen_producto
14,muskmelon,2912.19
2,Carambola,2916.05
9,Persimmon,2947.23


---
## KPI 10) Precio de venta medio de cada fruta (septiembre)

In [16]:
precio_medio_fruta = (
    venta_tipo.groupby("nombre", as_index=False)
    .agg(precio_medio=("precio_venta", "mean"))
    .sort_values("precio_medio", ascending=True)
)

precio_medio_fruta.columns = ["producto", "precio_medio"]
precio_medio_fruta

,producto,precio_medio
13,Tomatoes,3.451443
2,Carambola,3.481013
11,Plum,3.484037
0,Apple,3.486794
14,muskmelon,3.488549
1,Banana,3.495467
5,Mango,3.497049
3,Guava,3.501052
9,Persimmon,3.502102
12,Pomegranate,3.509057


---
## KPI 11) Fruta dañada por tipo (septiembre)

Ventas sin precio = fruta que no se pudo vender (dañada en la distribución).

In [17]:
venta_danada = venta_tipo[(venta_tipo["mes"] == 9) & (venta_tipo["precio_venta"].isna())]

fruta_danada_por_tipo = (
    venta_danada.groupby("nombre", as_index=False)
    .agg(nro_ventas_no_exitosas=("id_venta", "count"))
    .sort_values("nro_ventas_no_exitosas", ascending=False)
)

fruta_danada_por_tipo.columns = ["producto", "nro_ventas_no_exitosas"]
fruta_danada_por_tipo

,producto,nro_ventas_no_exitosas
3,Guava,189
0,Apple,109
4,Kiwi,81
5,Mango,41
8,Pear,39
1,Banana,35
10,Pitaya,24
14,muskmelon,24
12,Pomegranate,23
2,Carambola,22


---
## KPI 12) Pérdida total de la fruta dañada (septiembre)

In [18]:
venta_danada["coste"] = venta_danada["coste_inicial"] * venta_danada["peso"] / 1000

coste_ventas_no_exitosas = round(venta_danada["coste"].sum(), 2)
print(f"Pérdida total fruta dañada (euros): {coste_ventas_no_exitosas}")

Pérdida total fruta dañada (euros): 395.1


C:\Users\cesar\AppData\Local\Temp\ipykernel_12612\2127045389.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  venta_danada["coste"] = venta_danada["coste_inicial"] * venta_danada["peso"] / 1000


---
## KPI 13) Cuantía por tipo de fruta de los 5 clientes que más gastan

Primero identificamos los 5 clientes con más gasto (en todo el histórico con fecha válida, como en el SQL).
Después calculamos cuánto gastaron en cada fruta en septiembre.

In [19]:
# Top 5 clientes por gasto total (SQL: MONTH IS NOT NULL)
venta_con_fecha = venta[venta["mes"].notna()].copy()

top5_ids = (
    venta_con_fecha.groupby("id_cliente")["valor_euros"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .index
)

# Renombramos para evitar confusión con el nombre de la fruta
cliente_nombres = df_cliente.rename(columns={"nombre": "cliente"})

# Gasto por cliente y tipo de fruta en septiembre
venta_top5 = venta_tipo[
    (venta_tipo["mes"] == 9) & (venta_tipo["id_cliente"].isin(top5_ids))
].merge(cliente_nombres, on="id_cliente")

cuantia_top5_clientes = (
    venta_top5.groupby(["cliente", "nombre"], as_index=False)
    .agg(cuantia_gastada=("valor_euros", "sum"))
    .sort_values(["cliente", "cuantia_gastada"], ascending=[True, False])
)

cuantia_top5_clientes["cuantia_gastada"] = cuantia_top5_clientes["cuantia_gastada"].round(2)
cuantia_top5_clientes.columns = ["cliente", "tipo_fruta", "cuantia_gastada"]
cuantia_top5_clientes

,cliente,tipo_fruta,cuantia_gastada
3,Cosecha Fresca,Guava,637.91
0,Cosecha Fresca,Apple,351.50
4,Cosecha Fresca,Kiwi,241.46
5,Cosecha Fresca,Mango,137.45
1,Cosecha Fresca,Banana,80.43
...,...,...,...
73,SuperMercado Ideal,Tomatoes,69.22
70,SuperMercado Ideal,Pitaya,68.73
69,SuperMercado Ideal,Persimmon,64.57
72,SuperMercado Ideal,Pomegranate,63.92


---
## KPI 14) Porcentaje de beneficio por producto (septiembre)

In [20]:
venta_tipo["porcentaje_beneficio"] = (
    (venta_tipo["precio_venta"] - venta_tipo["coste_inicial"]) * 100 / venta_tipo["coste_inicial"]
)

beneficio_por_tipo = (
    venta_tipo.groupby("nombre", as_index=False)
    .agg(porcentaje_beneficio=("porcentaje_beneficio", "mean"))
    .sort_values("porcentaje_beneficio", ascending=False)
)

beneficio_por_tipo["porcentaje_beneficio"] = (
    beneficio_por_tipo["porcentaje_beneficio"].round(2).astype(str) + "%"
)
beneficio_por_tipo.columns = ["tipo_fruta", "porcentaje_beneficio"]
beneficio_por_tipo

,tipo_fruta,porcentaje_beneficio
2,Carambola,85.51%
12,Pomegranate,82.71%
10,Pitaya,82.59%
9,Persimmon,82.58%
8,Pear,82.1%
4,Kiwi,81.58%
1,Banana,81.48%
6,Orange,81.45%
7,Peach,81.35%
11,Plum,81.16%
